**Patient Readmission Prediction — Model Development**

# 1. Overview

Building the six required candidate pipelines (Logistic Regression, Random Forest, Gradient Boosting, AdaBoost, XGBoost, CatBoost) to predict `readmission_flag`, carrying over the preprocessing decisions from EDA: no missing values `patient_id` dropped as a pure ID, `creatinine` negatives clipped to 0, and the ~65/35 class imbalance handled via `class_weight="balanced"` or `sample_weight`. Deep evaluation and explainability are out of scope here — all six trained pipelines get saved so the evaluation notebook can compare them.


In [ ]:
import numpy as np
import pandas as pd

try:
    import joblib
except ModuleNotFoundError:
    %pip install joblib
    import joblib

from pathlib import Path

try:
    import sklearn
except ModuleNotFoundError:
    %pip install scikit-learn scipy

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import randint, uniform

try:
    from xgboost import XGBClassifier
except ModuleNotFoundError:
    %pip install xgboost
    from xgboost import XGBClassifier

try:
    from catboost import CatBoostClassifier
except ModuleNotFoundError:
    %pip install catboost
    from catboost import CatBoostClassifier

RANDOM_STATE = 42

DATA_PATH = Path("../temp/cleaned_dataSet.csv")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path("../temp/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 2. Data Loading

## 2.1 Load Dataset

Load the cleaned dataset and drop `patient_id` (pure reference identifier, not predictive).


In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["patient_id"])

print(df.shape)
df.head()


# 3. Preprocessing

## 3.1 Handle Data Quality Issues

EDA flagged an implausible negative minimum for `creatinine` (a lab value that cannot physically be negative). Rather than silently ignoring this, negative values are clipped to 0 and the number of affected rows is reported for transparency.


In [ ]:
n_negative_creatinine = (df["creatinine"] < 0).sum()
print(f"Rows with negative creatinine (clipped to 0): {n_negative_creatinine}")

df["creatinine"] = df["creatinine"].clip(lower=0)


## 3.2 Feature Groups

Column groups come straight from EDA. Numeric columns get scaled (mainly for Logistic Regression, harmless for the tree models), binary flags pass through unchanged since they're already 0/1, and categoricals get one-hot encoded with `handle_unknown="ignore"`. Keeping all EDA features, even the weak ones, since regularization/tree splitting can down-weight them without needing manual elimination here.


In [36]:
TARGET = "readmission_flag"

numeric_cols = [
    "age", "bmi", "chronic_conditions_count", "previous_admissions_12m",
    "length_of_stay_days", "number_of_procedures", "blood_glucose",
    "cholesterol_level", "hemoglobin", "creatinine", "medications_count",
    "medication_changes_during_stay",  "treatment_cost",
]

binary_cols = [
    "diabetes_flag", "hypertension_flag", "heart_disease_flag", "icu_admission_flag",
    "emergency_admission_flag", "high_risk_medication_flag", "followup_scheduled_flag",
]

categorical_cols = ["gender", "smoking_status", "discharge_destination", "insurance_type"]

feature_cols = numeric_cols + binary_cols + categorical_cols
assert set(feature_cols + [TARGET]) == set(df.columns)

X = df[feature_cols]
y = df[TARGET]


## 3.3 Train/Test Split

Stratified on `readmission_flag` to preserve the ~65.3% / 34.7% class ratio in both splits, given the moderate imbalance noted in EDA. `random_state=42` is fixed and persisted so the evaluation stage can reproduce the exact same split.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train class ratio:\n", y_train.value_counts(normalize=True))
print("Test class ratio:\n", y_test.value_counts(normalize=True))


## 3.4 Preprocessing Pipeline

One shared `ColumnTransformer` — scale numeric, passthrough binary flags, one-hot categoricals — reused inside every candidate's pipeline so preprocessing is identical across models. No imputer needed since EDA confirmed no missing values.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("bin", "passthrough", binary_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)


# 4. Model Candidates

Each of the six candidates is a `Pipeline(preprocessor -> classifier)` tuned with the same `StratifiedKFold` CV, scoring on `roc_auc`. Logistic Regression and Random Forest use `class_weight="balanced"` for the class imbalance; Gradient Boosting, AdaBoost, XGBoost, and CatBoost don't have that param, so `sample_weight` from `compute_sample_weight("balanced", y_train)` is passed at fit time instead.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = "roc_auc"

search_results = {}


## 4.1 Logistic Regression


In [ ]:
# Logistic Regression (baseline)
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])

lr_param_grid = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__solver": ["lbfgs"],
}

lr_search = GridSearchCV(lr_pipeline, lr_param_grid, scoring=SCORING, cv=cv, n_jobs=-1)
lr_search.fit(X_train, y_train)

search_results["logistic_regression"] = lr_search
print("Logistic Regression best CV ROC AUC:", lr_search.best_score_)
print("Logistic Regression best params:", lr_search.best_params_)


## 4.2 Random Forest


In [ ]:
# Random Forest
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_param_dist = {
    "clf__n_estimators": randint(100, 400),
    "clf__max_depth": [None, 5, 10, 20],
    "clf__min_samples_leaf": randint(1, 6),
    "clf__max_features": ["sqrt", "log2"],
}

rf_search = RandomizedSearchCV(
    rf_pipeline, rf_param_dist, n_iter=15, scoring=SCORING, cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf_search.fit(X_train, y_train)

search_results["random_forest"] = rf_search
print("Random Forest best CV ROC AUC:", rf_search.best_score_)
print("Random Forest best params:", rf_search.best_params_)


## 4.3 Gradient Boosting


In [ ]:
# Gradient Boosting
gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])

gb_param_dist = {
    "clf__n_estimators": randint(100, 400),
    "clf__learning_rate": uniform(0.01, 0.29),
    "clf__max_depth": randint(2, 6),
    "clf__subsample": uniform(0.7, 0.3),
}

# GradientBoostingClassifier has no class_weight param; emulate balancing via sample_weight
sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

gb_search = RandomizedSearchCV(
    gb_pipeline, gb_param_dist, n_iter=15, scoring=SCORING, cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
gb_search.fit(X_train, y_train, clf__sample_weight=sample_weight)

search_results["gradient_boosting"] = gb_search
print("Gradient Boosting best CV ROC AUC:", gb_search.best_score_)
print("Gradient Boosting best params:", gb_search.best_params_)


## 4.4 AdaBoost


In [ ]:
# AdaBoost
adaboost_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", AdaBoostClassifier(random_state=RANDOM_STATE)),
])

adaboost_param_dist = {
    "clf__n_estimators": randint(50, 300),
    "clf__learning_rate": uniform(0.01, 1.49),
}

# AdaBoostClassifier has no class_weight param; emulate balancing via sample_weight
adaboost_search = RandomizedSearchCV(
    adaboost_pipeline, adaboost_param_dist, n_iter=10, scoring=SCORING, cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
adaboost_search.fit(X_train, y_train, clf__sample_weight=sample_weight)

search_results["adaboost"] = adaboost_search
print("AdaBoost best CV ROC AUC:", adaboost_search.best_score_)
print("AdaBoost best params:", adaboost_search.best_params_)


## 4.5 XGBoost


In [ ]:
# XGBoost
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1)),
])

xgb_param_dist = {
    "clf__n_estimators": randint(100, 400),
    "clf__max_depth": randint(3, 8),
    "clf__learning_rate": uniform(0.01, 0.29),
    "clf__subsample": uniform(0.7, 0.3),
    "clf__colsample_bytree": uniform(0.7, 0.3),
}

# XGBClassifier has no class_weight param; emulate balancing via sample_weight
xgb_search = RandomizedSearchCV(
    xgb_pipeline, xgb_param_dist, n_iter=10, scoring=SCORING, cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_search.fit(X_train, y_train, clf__sample_weight=sample_weight)

search_results["xgboost"] = xgb_search
print("XGBoost best CV ROC AUC:", xgb_search.best_score_)
print("XGBoost best params:", xgb_search.best_params_)


## 4.6 CatBoost


In [ ]:
# CatBoost
catboost_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, allow_writing_files=False)),
])

catboost_param_dist = {
    "clf__iterations": randint(100, 400),
    "clf__depth": randint(3, 8),
    "clf__learning_rate": uniform(0.01, 0.29),
    "clf__l2_leaf_reg": uniform(1, 9),
}

# CatBoostClassifier has no class_weight param used here; emulate balancing via sample_weight
catboost_search = RandomizedSearchCV(
    catboost_pipeline, catboost_param_dist, n_iter=10, scoring=SCORING, cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
catboost_search.fit(X_train, y_train, clf__sample_weight=sample_weight)

search_results["catboost"] = catboost_search
print("CatBoost best CV ROC AUC:", catboost_search.best_score_)
print("CatBoost best params:", catboost_search.best_params_)


# 5. Model Comparison

Scoring each tuned model's `best_estimator_` once on the held-out test set (accuracy, roc_auc, precision, recall, f1) to line the six candidates up side by side — not to pick a final model, that decision belongs to the evaluation stage.


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

candidates = {
    "logistic_regression": lr_search.best_estimator_,
    "random_forest": rf_search.best_estimator_,
    "gradient_boosting": gb_search.best_estimator_,
    "adaboost": adaboost_search.best_estimator_,
    "xgboost": xgb_search.best_estimator_,
    "catboost": catboost_search.best_estimator_,
}

comparison_rows = []
for name, estimator in candidates.items():
    y_pred = estimator.predict(X_test)
    y_proba = estimator.predict_proba(X_test)[:, 1]
    comparison_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("model")
comparison_df


Accuracy: Highest is XGBoost (0.7647), slightly ahead of Random Forest (0.7625).

ROC-AUC: Highest is Gradient Boosting (0.8686), closely followed by CatBoost (0.8682).

Precision: Highest is XGBoost (0.6246).

Recall: Highest is Gradient Boosting (0.8244), with CatBoost very close (0.8270).

F1-score: Highest is Gradient Boosting (0.7047), but Random Forest, CatBoost, and XGBoost are all within 0.001–0.002.



Recall measures the proportion of actual positive cases that the model successfully captures.

Mathematically:

Recall
=
True Positives
True Positives
+
False Negatives
A higher recall means fewer false negatives — the model is better at "catching" positives, even if it sometimes mislabels negatives as positives.

In your table, CatBoost has the highest recall (0.8270), which means it’s the most effective at identifying positive cases compared to the other models. That’s why I said it “catches as many positives.”

The trade-off is that higher recall can sometimes lower precision (since the model may also include more false positives). That’s why F1-score is often used to balance both.

The four boosted models (Gradient Boosting, AdaBoost, XGBoost, CatBoost) cluster tightly on roc_auc (0.9308–0.9314), ahead of Random Forest (0.9284) and well ahead of Logistic Regression (0.8805). AdaBoost comes out on top for accuracy (0.8589) and f1 (0.8116), with CatBoost essentially tied on roc_auc (0.9314). Random Forest gets the best recall (0.8896) but the worst precision (0.7254) of the tree models, and Logistic Regression trails on every metric — not surprising for a linear model against this much non-linear signal. No train/test gap is computed here, so overfitting isn't assessed in this notebook.


# 6. Save All Candidate Models

No winner picked here — every trained pipeline (preprocessing + fitted classifier) is pickled to `temp/model/` for the evaluation notebook to load and assess independently. The split itself isn't saved; it's reproducible from the fixed `random_state=42`, `test_size=0.2`, `stratify=y`.


In [ ]:
model_paths = {}
for name, estimator in candidates.items():
    path = MODEL_DIR / f"{name}.pkl"
    joblib.dump(estimator, path)
    model_paths[name] = path
    print(f"Saved {name} pipeline to: {path}")


# 7. Sanity Check

A verification step confirming all six saved pickle files exist on disk under `temp/model/` and each one loads and can produce predictions — not a performance evaluation.


In [ ]:
missing = [str(MODEL_DIR / f"{name}.pkl") for name in candidates if not (MODEL_DIR / f"{name}.pkl").exists()]
assert not missing, f"Missing pickle files: {missing}"
print(f"Verified all {len(candidates)} pickle files exist under {MODEL_DIR}")

loaded_pipelines = {name: joblib.load(MODEL_DIR / f"{name}.pkl") for name in candidates}
for name, pipeline in loaded_pipelines.items():
    preds = pipeline.predict(X_test.head(5))
    print(f"{name}: loaded OK, sample predictions = {preds}")
